---

## Part B – Loops, Memory & Human-in-the-Loop

### Loops (Cycles)

LangGraph supports **cycles** — a node can have an edge pointing back to an earlier node, enabling:
- Agent ↔ tools loops
- Iterative refinement
- Retry logic

```
START  →  [agent]  ──  has tool_calls?  ──→  YES  →  [tools]  ──┐
                    └─  NO  →  END                                └──→  back to [agent]
```

The `Annotated[list[BaseMessage], operator.add]` annotation tells LangGraph to **append** updates to the list instead of replacing it — essential for accumulating a conversation history across loop iterations.

In [ ]:
import os
import operator
import math
import datetime
from typing import TypedDict, Annotated, Literal
from dotenv import load_dotenv

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage,
)
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(override=True)
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
if not anthropic_api_key:
    raise ValueError("ANTHROPIC_API_KEY not found. Add it to your .env file.")

chat_model = ChatAnthropic(
    model="claude-opus-4-6",
    temperature=0.3,
    anthropic_api_key=anthropic_api_key,
)

In [ ]:
#Gateway API Key and Base URL
# ──────────────────────────────────────────────────────────────
# 1. ENV SETUP
# ──────────────────────────────────────────────────────────────
import os
from dotenv import load_dotenv
import datetime
import math
load_dotenv(override=True)

llmgw_api_key = os.getenv("LLMGW_API_KEY")
llmgw_base_url = os.getenv("LLMGW_BASE_URL", "https://llmgw-wp.tekstac.com")

if not llmgw_api_key:
    raise ValueError("LLMGW_API_KEY not found. Add it to your .env file.")

# Set Anthropic-compatible env vars
os.environ["ANTHROPIC_API_KEY"] = llmgw_api_key
os.environ["ANTHROPIC_BASE_URL"] = llmgw_base_url

print("=" * 50)
print("ENVIRONMENT STATUS")
print("=" * 50)
print(f"LLMGW_API_KEY  : {'✅ set' if llmgw_api_key else '❌ missing'}")
print(f"LLMGW_BASE_URL : {llmgw_base_url}")
print("=" * 50)


# ──────────────────────────────────────────────────────────────
# 2. IMPORTS
# ──────────────────────────────────────────────────────────────
import operator
from typing import TypedDict, Annotated, Literal
from IPython.display import Image,display

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END

chat_model = ChatAnthropic(
    model="global.anthropic.claude-sonnet-4-6",
    temperature=0.3,
    anthropic_api_key=llmgw_api_key,
    base_url=llmgw_base_url,
)


State["messages"] = [
    HumanMessage("What is 47 + 89?"),
    AIMessage(...tool call...),
    ToolMessage("136")
]


In [ ]:
print("PART B – Cyclic Agent Graph")
print("=" * 50)

# ── Tools ───────────────────────────────────────────
@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together"""
    return a + b


@tool
def get_current_date() -> str:
    """Return current date in YYYY-MM-DD format"""
    return datetime.date.today().isoformat()


@tool
def calculate_square_root(number: float) -> float:
    """Calculate square root of a number"""
    return round(math.sqrt(number), 4)


agent_tools = [add_numbers, get_current_date, calculate_square_root]
tool_map = {t.name: t for t in agent_tools}
llm_with_tools = chat_model.bind_tools(agent_tools)


# ── State ───────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]


# ── LLM Node ────────────────────────────────────────
def agent_llm_node(state: AgentState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


# ── Tool Execution Node ─────────────────────────────
def tools_exec_node(state: AgentState):
    last_msg = state["messages"][-1]
    results = []

    for tc in last_msg.tool_calls:
        output = tool_map[tc["name"]].invoke(tc["args"])
        results.append(
            ToolMessage(
                content=str(output),
                tool_call_id=tc["id"]
            )
        )

    return {"messages": results}


# ── Routing ─────────────────────────────────────────
def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    last_msg = state["messages"][-1]

    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"

    return END


# ── Build Graph ─────────────────────────────────────
builder = StateGraph(AgentState)

builder.add_node("agent", agent_llm_node)
builder.add_node("tools", tools_exec_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue)
builder.add_edge("tools", "agent")

graph = builder.compile()


# ── Run Example ─────────────────────────────────────
queries = [
    "What is 47 plus 89?",
    "What is today's date and the square root of 225?"
]

for q in queries:
    print("\n" + "-" * 40)
    print("User:", q)

    result = graph.invoke({
        "messages": [HumanMessage(content=q)]
    })

    print("Answer:", result["messages"][-1].content)

    print("\nMessages Flow:")
    for msg in result["messages"]:
        print(type(msg).__name__, "->", str(msg.content)[:60])

In [ ]:
# ── Cyclic Agent Graph: LLM \u2194 Tools Loop ────────────────────────────────
print("\U0001f504 PART B \u2013 Loops: Cyclic Agent Graph")
print("=" * 55)

# ── Tool definitions ──────────────────────────────────────────────────────
@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together and return the result."""
    return a + b

@tool
def get_current_date() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return datetime.date.today().isoformat()

@tool
def calculate_square_root(number: float) -> float:
    """Calculate the square root of a non-negative number."""
    if number < 0:
        return "Error: cannot compute square root of a negative number."
    return round(math.sqrt(number), 4)

agent_tools    = [add_numbers, get_current_date, calculate_square_root]
tool_map       = {t.name: t for t in agent_tools}
llm_with_tools = chat_model.bind_tools(agent_tools)

# ── State: messages ACCUMULATE via operator.add ───────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

# ── Nodes ─────────────────────────────────────────────────────────────────
def agent_llm_node(state: AgentState) -> dict:
    """Ask the LLM what to do next. Returns a tool call or a final answer."""
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

def tools_exec_node(state: AgentState) -> dict:
    """Execute every tool call the LLM requested."""
    last_msg = state["messages"][-1]
    results  = []
    for tc in last_msg.tool_calls:
        output = tool_map[tc["name"]].invoke(tc["args"])
        results.append(ToolMessage(content=str(output), tool_call_id=tc["id"]))
    return {"messages": results}

# ── Routing: loop or stop ─────────────────────────────────────────────────
def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    """If the last message has tool calls -> tools; otherwise -> END."""
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

# ── Build cyclic graph ────────────────────────────────────────────────────
agent_builder = StateGraph(AgentState)
agent_builder.add_node("agent", agent_llm_node)
agent_builder.add_node("tools", tools_exec_node)

agent_builder.add_edge(START, "agent")
agent_builder.add_conditional_edges("agent", should_continue)
agent_builder.add_edge("tools", "agent")        # \u2190 CYCLE: tools feeds back to agent

agent_graph = agent_builder.compile()

print("Graph: START \u2192 agent \u2192 (tool_calls?) \u2192 tools \u2192 agent \u2026 OR \u2192 END\n")

for query in [
    "What is 47 plus 89?",
    "What is today's date and the square root of 225?",
]:
    print("\u2500" * 52)
    print(f"\U0001f464 Query : {query}")
    r = agent_graph.invoke({"messages": [HumanMessage(content=query)]})
    print(f"\U0001f916 Answer: {r['messages'][-1].content.strip()}")
    print(f"   Trace : {len(r['messages'])} messages exchanged\n")
    print("   Messages:")
    for i, msg in enumerate(r['messages'], 1):
        print(f"   {i}. {type(msg).__name__}: {str(msg.content)[:80]}")
    print()


In [ ]:
# Full Program (Single Script)
# ──────────────────────────────────────────────────────────────
# 1. ENV SETUP
# ──────────────────────────────────────────────────────────────
import os
from dotenv import load_dotenv

load_dotenv(override=True)

llmgw_api_key = os.getenv("LLMGW_API_KEY")
llmgw_base_url = os.getenv("LLMGW_BASE_URL", "https://llmgw-wp.tekstac.com")

if not llmgw_api_key:
    raise ValueError("LLMGW_API_KEY not found. Add it to your .env file.")

# Set Anthropic-compatible env vars
os.environ["ANTHROPIC_API_KEY"] = llmgw_api_key
os.environ["ANTHROPIC_BASE_URL"] = llmgw_base_url

print("=" * 50)
print("ENVIRONMENT STATUS")
print("=" * 50)
print(f"LLMGW_API_KEY  : {'✅ set' if llmgw_api_key else '❌ missing'}")
print(f"LLMGW_BASE_URL : {llmgw_base_url}")
print("=" * 50)


# ──────────────────────────────────────────────────────────────
# 2. IMPORTS
# ──────────────────────────────────────────────────────────────
import operator
from typing import TypedDict, Annotated, Literal

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END


# ──────────────────────────────────────────────────────────────
# 3. MOCK DATABASE (Expanded)
# ──────────────────────────────────────────────────────────────
ORDER_DB = {
    "ORD123": {"status": "Shipped", "item": "Wireless Headphones", "expected_delivery": "Tomorrow"},
    "ORD456": {"status": "Processing", "item": "Mechanical Keyboard", "expected_delivery": "Next Tuesday"},
    "ORD789": {"status": "Delivered", "item": "Gaming Mouse", "expected_delivery": "Delivered Yesterday"},
    "ORD321": {"status": "Out for Delivery", "item": "Laptop Stand", "expected_delivery": "Today"},
    "ORD654": {"status": "Cancelled", "item": "Smart Watch", "expected_delivery": "N/A"},
    "ORD999": {"status": "Delayed", "item": "Bluetooth Speaker", "expected_delivery": "In 3 days"},
}


# ──────────────────────────────────────────────────────────────
# 4. TOOLS
# ──────────────────────────────────────────────────────────────
@tool
def check_order_status(order_id: str) -> str:
    """Check the shipping status and expected delivery of an order."""
    print(f"[Tool] Checking DB for {order_id}")
    order = ORDER_DB.get(order_id.upper())

    if order:
        return (
            f"Order {order_id}: {order['item']} is '{order['status']}'. "
            f"Expected delivery: {order['expected_delivery']}."
        )
    return f"Order {order_id} not found. Please verify the order ID."


@tool
def calculate_refund_eligibility(days_since_purchase: int) -> str:
    """Check refund eligibility based on purchase duration."""
    print(f"[Tool] Refund check for {days_since_purchase} days")

    if days_since_purchase <= 30:
        return "Eligible for full refund."
    elif days_since_purchase <= 60:
        return "Eligible for store credit."
    else:
        return "Not eligible for refund."


# ──────────────────────────────────────────────────────────────
# 5. MODEL SETUP
# ──────────────────────────────────────────────────────────────
agent_tools = [check_order_status, calculate_refund_eligibility]
tool_map = {tool.name: tool for tool in agent_tools}

chat_model = ChatAnthropic(
    model="global.anthropic.claude-sonnet-4-6",
    temperature=0.3,
    anthropic_api_key=llmgw_api_key,
    base_url=llmgw_base_url,
)

llm_with_tools = chat_model.bind_tools(agent_tools)

# Connection test
try:
    test = chat_model.invoke("Capital of France?")
    print(f"\n✅ Connection OK: {test.content}")
except Exception as e:
    print(f"❌ Connection Error: {e}")


# ──────────────────────────────────────────────────────────────
# 6. AGENT STATE
# ──────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]


# ──────────────────────────────────────────────────────────────
# 7. GRAPH NODES
# ──────────────────────────────────────────────────────────────
def support_agent_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


def tool_execution_node(state: AgentState):
    last_message = state["messages"][-1]
    results = []

    for tool_call in last_message.tool_calls:
        tool_output = tool_map[tool_call["name"]].invoke(tool_call["args"])

        results.append(
            ToolMessage(
                content=str(tool_output),
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": results}


# ──────────────────────────────────────────────────────────────
# 8. ROUTING LOGIC
# ──────────────────────────────────────────────────────────────
def route_next_step(state: AgentState) -> Literal["tools", "__end__"]:
    last_message = state["messages"][-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    return END


# ──────────────────────────────────────────────────────────────
# 9. BUILD GRAPH
# ──────────────────────────────────────────────────────────────
workflow = StateGraph(AgentState)

workflow.add_node("agent", support_agent_node)
workflow.add_node("tools", tool_execution_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", route_next_step)
workflow.add_edge("tools", "agent")

app = workflow.compile()


# ──────────────────────────────────────────────────────────────
# 10. RUN TEST QUERIES
# ──────────────────────────────────────────────────────────────
queries = [
    "Hi, where is my order ORD123?",
    "Is ORD321 delivered yet?",
    "I bought something 45 days ago, can I get refund?",
    "Check ORD999 and also tell refund for 10 days purchase",
    "What about order ORD888?",  # Not in DB
]

print("\n--- Support Bot Running ---")

for query in queries:
    print(f"\nUser: {query}")

    result = app.invoke({
        "messages": [HumanMessage(content=query)]
    })

    final_message = result["messages"][-1].content
    print(f"Bot: {final_message}")

## Replace Dictionary with MySQL

### Create Table in MySQL Workbench
Run this in your DB:

CREATE DATABASE support_bot;

USE support_bot;

CREATE TABLE orders (
    order_id VARCHAR(10) PRIMARY KEY,
    item VARCHAR(100),
    status VARCHAR(50),
    expected_delivery VARCHAR(50)
);


### Insert Sample Data

INSERT INTO orders VALUES
('ORD123', 'Wireless Headphones', 'Shipped', 'Tomorrow'),
('ORD456', 'Mechanical Keyboard', 'Processing', 'Next Tuesday'),
('ORD789', 'Gaming Mouse', 'Delivered', 'Yesterday'),
('ORD321', 'Laptop Stand', 'Out for Delivery', 'Today'),
('ORD654', 'Smart Watch', 'Cancelled', 'N/A'),
('ORD999', 'Bluetooth Speaker', 'Delayed', 'In 3 days');

### Add Multiple Rows

INSERT INTO orders (order_id, item, status, expected_delivery)
VALUES
('ORD888', 'Laptop Bag', 'Shipped', 'Tomorrow'),
('ORD889', 'Monitor', 'Processing', 'Next Friday'),
('ORD890', 'Webcam', 'Delivered', 'Yesterday');

### Update Existing Data
UPDATE orders
SET status = 'Delivered', expected_delivery = 'Today'
WHERE order_id = 'ORD888';

### Delete Row (optional)
DELETE FROM orders WHERE order_id = 'ORD905';

#### Install MySQL connector

In [ ]:
%pip install mysql-connector-python

#### Add DB Connection in Python

In [ ]:

import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_password",  #eg Tek@12345
    database="support_bot"
)

cursor = conn.cursor(dictionary=True)

In [ ]:
# Full Program (Single Script) with MySQL Integration
# ──────────────────────────────────────────────────────────────
# 1. ENV SETUP
# ──────────────────────────────────────────────────────────────
import os
from dotenv import load_dotenv

load_dotenv(override=True)

llmgw_api_key = os.getenv("LLMGW_API_KEY")
llmgw_base_url = os.getenv("LLMGW_BASE_URL", "https://llmgw-wp.tekstac.com")

if not llmgw_api_key:
    raise ValueError("LLMGW_API_KEY not found. Add it to your .env file.")

# Set Anthropic-compatible env vars
os.environ["ANTHROPIC_API_KEY"] = llmgw_api_key
os.environ["ANTHROPIC_BASE_URL"] = llmgw_base_url

print("=" * 50)
print("ENVIRONMENT STATUS")
print("=" * 50)
print(f"LLMGW_API_KEY  : {'✅ set' if llmgw_api_key else '❌ missing'}")
print(f"LLMGW_BASE_URL : {llmgw_base_url}")
print("=" * 50)


# ──────────────────────────────────────────────────────────────
# 2. IMPORTS
# ──────────────────────────────────────────────────────────────
import operator
from typing import TypedDict, Annotated, Literal

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END





# ──────────────────────────────────────────────────────────────
# 4. TOOLS
# ──────────────────────────────────────────────────────────────
@tool
def check_order_status(order_id: str) -> str:
    """Check order status from MySQL database"""

    print(f"[Tool] Fetching order {order_id} from DB")

    query = "SELECT * FROM orders WHERE order_id = %s"
    cursor.execute(query, (order_id.upper(),))
    order = cursor.fetchone()

    if order:
        return (
            f"Order {order_id}: {order['item']} is '{order['status']}'. "
            f"Expected delivery: {order['expected_delivery']}."
        )

    return f"Order {order_id} not found."
@tool
def calculate_refund_eligibility(days_since_purchase: int) -> str:
    """Check refund eligibility based on purchase duration."""
    print(f"[Tool] Refund check for {days_since_purchase} days")

    if days_since_purchase <= 30:
        return "Eligible for full refund."
    elif days_since_purchase <= 60:
        return "Eligible for store credit."
    else:
        return "Not eligible for refund."


# ──────────────────────────────────────────────────────────────
# 5. MODEL SETUP
# ──────────────────────────────────────────────────────────────
agent_tools = [check_order_status, calculate_refund_eligibility]
tool_map = {tool.name: tool for tool in agent_tools}

chat_model = ChatAnthropic(
    model="global.anthropic.claude-sonnet-4-6",
    temperature=0.3,
    anthropic_api_key=llmgw_api_key,
    base_url=llmgw_base_url,
)

llm_with_tools = chat_model.bind_tools(agent_tools)



# ──────────────────────────────────────────────────────────────
# 6. AGENT STATE
# ──────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]


# ──────────────────────────────────────────────────────────────
# 7. GRAPH NODES
# ──────────────────────────────────────────────────────────────
def support_agent_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


def tool_execution_node(state: AgentState):
    last_message = state["messages"][-1]
    results = []

    for tool_call in last_message.tool_calls:
        tool_output = tool_map[tool_call["name"]].invoke(tool_call["args"])

        results.append(
            ToolMessage(
                content=str(tool_output),
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": results}


# ──────────────────────────────────────────────────────────────
# 8. ROUTING LOGIC
# ──────────────────────────────────────────────────────────────
def route_next_step(state: AgentState) -> Literal["tools", "__end__"]:
    last_message = state["messages"][-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    return END


# ──────────────────────────────────────────────────────────────
# 9. BUILD GRAPH
# ──────────────────────────────────────────────────────────────
workflow = StateGraph(AgentState)

workflow.add_node("agent", support_agent_node)
workflow.add_node("tools", tool_execution_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", route_next_step)
workflow.add_edge("tools", "agent")

app = workflow.compile()



In [ ]:
result = app.invoke({
        "messages": [HumanMessage(content="Where is my order ORD123")]
    })

final_message = result["messages"][-1].content
print(f"Bot: {final_message}")

In [ ]:
result = app.invoke({
        "messages": [HumanMessage(content="Status of my webcam order (ORD890) please")]
    })

final_message = result["messages"][-1].content
print(f"Bot: {final_message}")

### Memory with `MemorySaver` and `thread_id`

By default every `.invoke()` call is **stateless** — the graph forgets everything. Attaching a **checkpointer** enables persistence across calls.

| Config | Behaviour |
|--------|-----------|
| Same `thread_id` | State is restored → graph remembers previous turns |
| Different `thread_id` | Fresh start → no memory of other sessions |

```python
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "alice-session"}}
graph.invoke({"messages": [HumanMessage("My name is Alice.")]}, config=config)
graph.invoke({"messages": [HumanMessage("What is my name?")]}, config=config)
# ↑ Claude remembers "Alice" because the state was checkpointed ✓
```

`MemorySaver` is in-process (process memory). For production use `SqliteSaver` or `PostgresSaver`.

In [ ]:

from langgraph.checkpoint.memory import MemorySaver
# ── Persistent Memory with MemorySaver + thread_id ───────────────────────
print("\U0001f9e0 Memory / Checkpoints with MemorySaver")
print("=" * 55)

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

def chat_node(state: ChatState) -> dict:
    return {"messages": [chat_model.invoke(state["messages"])]}
#"messages": [HumanMessage(...), AIMessage(...), ...]}

chat_builder = StateGraph(ChatState)
chat_builder.add_node("chat", chat_node)
chat_builder.add_edge(START, "chat")
chat_builder.add_edge("chat", END)

# ── Attach checkpointer ───────────────────────────────────────────────────
mem_saver     = MemorySaver()
persistent_chat = chat_builder.compile(checkpointer=mem_saver)

# ── Sachin's session ───────────────────────────────────────────────────────
sachin_cfg  = {"configurable": {"thread_id": "sachin-001"}}
sachin_turns = [
    "Hi! My name is Sachin and I am a cloud architect at a fintech firm.",
    "I mainly use AWS and Kubernetes day to day.",
    "What do you know about me so far?",    # tests memory recall
]

print("\u2500\u2500 Sachin's session (thread_id = sachin-001) \u2500\u2500")
for msg in sachin_turns:
    r = persistent_chat.invoke({"messages": [HumanMessage(content=msg)]}, config=sachin_cfg)
    reply = r["messages"][-1].content.strip()
    print(f"\U0001f464 Sachin : {msg}")
    print(f"\U0001f916 AI    : {reply[:210]}")
    print()

# ── Rohith's session (different thread_id \u2192 fresh memory) ────────────────────
rohith_cfg = {"configurable": {"thread_id": "rohith-001"}}
r_rohith   = persistent_chat.invoke(
    {"messages": [HumanMessage(content="What is my name?")]}, config=rohith_cfg
)
print("\u2500\u2500 Rohith's session (thread_id = rohith-001 \u2014 fresh memory) \u2500\u2500")
print(f"\U0001f464 Rohith : What is my name?")
print(f"\U0001f916 AI  : {r_rohith['messages'][-1].content.strip()[:210]}")


### Streamlit app for order and refund status





import os
import operator
from typing import TypedDict, Annotated, Literal

import streamlit as st
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, ToolMessage, BaseMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# ──────────────────────────────────────────────────────────────
# 1. PAGE CONFIG
# ──────────────────────────────────────────────────────────────
st.set_page_config(page_title="Order Support Bot",  layout="centered")

# ──────────────────────────────────────────────────────────────
# 2. ENV SETUP
# ──────────────────────────────────────────────────────────────
load_dotenv(override=True)

LLMGW_API_KEY = os.getenv("LLMGW_API_KEY")
LLMGW_BASE_URL = os.getenv("LLMGW_BASE_URL", "https://llmgw-wp.tekstac.com")

# Optional: define these in your .env for convenience
# MYSQL_HOST=localhost
# MYSQL_PORT=3306
# MYSQL_USER=root
# MYSQL_PASSWORD=your_password
# MYSQL_DATABASE=support_bot
MYSQL_HOST = os.getenv("MYSQL_HOST", "localhost")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_USER = os.getenv("MYSQL_USER", "root")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "Tek@12345")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE", "support_bot")

if not LLMGW_API_KEY:
    st.error("LLMGW_API_KEY not found. Add it to your .env file.")
    st.stop()

# Anthropic-compatible env vars for LangChain internals
os.environ["ANTHROPIC_API_KEY"] = LLMGW_API_KEY
os.environ["ANTHROPIC_BASE_URL"] = LLMGW_BASE_URL

# ──────────────────────────────────────────────────────────────
# 3. DB HELPERS
# ──────────────────────────────────────────────────────────────
def get_db_connection():
    """Create and return a MySQL connection."""
    return mysql.connector.connect(
        host=MYSQL_HOST,
        port=MYSQL_PORT,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD,
        database=MYSQL_DATABASE,
    )


def init_db():
    """Create the orders table if it does not exist."""
    try:
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute(
            """
            CREATE TABLE IF NOT EXISTS orders (
                order_id VARCHAR(20) PRIMARY KEY,
                item VARCHAR(255) NOT NULL,
                status VARCHAR(100) NOT NULL,
                expected_delivery VARCHAR(100) NOT NULL
            )
            """
        )
        conn.commit()
    finally:
        try:
            cursor.close()
            conn.close()
        except Exception:
            pass


def seed_sample_data_if_empty():
    """Insert sample rows only when the table is empty."""
    sample_rows = [
        ("ORD123", "Wireless Headphones", "Shipped", "Tomorrow"),
        ("ORD456", "Mechanical Keyboard", "Processing", "Next Tuesday"),
        ("ORD789", "Gaming Mouse", "Delivered", "Yesterday"),
        ("ORD321", "Laptop Stand", "Out for Delivery", "Today"),
        ("ORD654", "Smart Watch", "Cancelled", "N/A"),
        ("ORD999", "Bluetooth Speaker", "Delayed", "In 3 days"),
        ("ORD777", "USB Hub", "Processing", "Next Monday"),
        ("ORD888", "Laptop Bag", "Shipped", "Tomorrow"),
    ]
    try:
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM orders")
        count = cursor.fetchone()[0]
        if count == 0:
            cursor.executemany(
                "INSERT INTO orders (order_id, item, status, expected_delivery) VALUES (%s, %s, %s, %s)",
                sample_rows,
            )
            conn.commit()
    finally:
        try:
            cursor.close()
            conn.close()
        except Exception:
            pass


def get_all_orders(limit: int = 100):
    try:
        conn = get_db_connection()
        cursor = conn.cursor(dictionary=True)
        cursor.execute(
            "SELECT order_id, item, status, expected_delivery FROM orders ORDER BY order_id LIMIT %s",
            (limit,),
        )
        rows = cursor.fetchall()
        return rows
    finally:
        try:
            cursor.close()
            conn.close()
        except Exception:
            pass

# ──────────────────────────────────────────────────────────────
# 4. TOOLS
# ──────────────────────────────────────────────────────────────
@tool
def check_order_status(order_id: str) -> str:
    """Check the shipping status and expected delivery for a specific customer order. Provide the order ID such as ORD123."""
    try:
        conn = get_db_connection()
        cursor = conn.cursor(dictionary=True)
        query = "SELECT order_id, item, status, expected_delivery FROM orders WHERE order_id = %s"
        cursor.execute(query, (order_id.upper(),))
        order = cursor.fetchone()

        if order:
            return (
                f"Order {order['order_id']}: {order['item']} is currently '{order['status']}'. "
                f"Expected delivery: {order['expected_delivery']}."
            )
        return f"Order {order_id.upper()} not found in the database. Please verify the order ID."
    except Error as e:
        return f"Database error while checking order {order_id.upper()}: {e}"
    finally:
        try:
            cursor.close()
            conn.close()
        except Exception:
            pass


@tool
def calculate_refund_eligibility(days_since_purchase: int) -> str:
    """Determine refund eligibility based on how many days ago the customer purchased the item."""
    if days_since_purchase <= 30:
        return "Eligible for a full refund."
    elif days_since_purchase <= 60:
        return "Eligible for store credit only."
    else:
        return "Not eligible for a refund. The 60-day return window has expired."



# ──────────────────────────────────────────────────────────────
# 5. LLM + GRAPH SETUP
# ──────────────────────────────────────────────────────────────
@st.cache_resource(show_spinner=False)
def build_chatbot():
    chat_model = ChatAnthropic(
        model="global.anthropic.claude-sonnet-4-6",
        temperature=0.3,
        anthropic_api_key=LLMGW_API_KEY,
        base_url=LLMGW_BASE_URL,
    )

    agent_tools = [check_order_status, calculate_refund_eligibility]
    tool_map = {t.name: t for t in agent_tools}
    llm_with_tools = chat_model.bind_tools(agent_tools)

    class AgentState(TypedDict):
        messages: Annotated[list[BaseMessage], operator.add]

    def support_agent_node(state: AgentState):
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    def tool_execution_node(state: AgentState):
        last_message = state["messages"][-1]
        results = []

        for tool_call in last_message.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_output = tool_map[tool_name].invoke(tool_args)
            results.append(
                ToolMessage(
                    content=str(tool_output),
                    tool_call_id=tool_call["id"],
                )
            )
        return {"messages": results}

    def route_next_step(state: AgentState) -> Literal["tools", "__end__"]:
        last_message = state["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            return "tools"
        return END

    workflow = StateGraph(AgentState)
    workflow.add_node("agent", support_agent_node)
    workflow.add_node("tools", tool_execution_node)
    workflow.add_edge(START, "agent")
    workflow.add_conditional_edges("agent", route_next_step)
    workflow.add_edge("tools", "agent")

    mem_saver = MemorySaver()
    app = workflow.compile(checkpointer=mem_saver)
    return app



# ──────────────────────────────────────────────────────────────
# 6. STREAMLIT SESSION STATE
# ──────────────────────────────────────────────────────────────
if "thread_id" not in st.session_state:
    # Stable per browser session; change it manually in sidebar if needed
    st.session_state.thread_id = "streamlit-user-001"

if "chat_messages" not in st.session_state:
    st.session_state.chat_messages = [
        {
            "role": "assistant",
            "content": (
                "Hi! I’m your order support bot. You can ask things like:\n\n"
                "- Where is my order ORD123?\n"
                "- Check order ORD456\n"
                "- I bought something 45 days ago, can I get a refund?"
            ),
        }
    ]


# ──────────────────────────────────────────────────────────────
# 7. INITIALIZE APP RESOURCES
# ──────────────────────────────────────────────────────────────
try:
    init_db()
    seed_sample_data_if_empty()
except Error as db_init_error:
    st.error(f"Database initialization failed: {db_init_error}")
    st.stop()

app = build_chatbot()


# ──────────────────────────────────────────────────────────────
# 8. SIDEBAR
# ──────────────────────────────────────────────────────────────
with st.sidebar:
    st.header("Settings")
    st.caption("Configure memory thread and verify environment/database.")

    thread_id_input = st.text_input("Thread ID", value=st.session_state.thread_id, help="Same thread_id = same conversation memory")
    if thread_id_input != st.session_state.thread_id:
        st.session_state.thread_id = thread_id_input.strip() or "streamlit-user-001"
        st.session_state.chat_messages = [
            {
                "role": "assistant",
                "content": "Thread changed. This is a fresh visible chat window. Memory for each thread is isolated.",
            }
        ]

    if st.button("Clear visible chat"):
        st.session_state.chat_messages = [
            {
                "role": "assistant",
                "content": "Visible chat cleared. If you keep the same thread_id, LangGraph memory for that thread still exists in this app process.",
            }
        ]

    st.divider()
    st.subheader("Environment")
    st.write(f"**LLMGW Base URL:** {LLMGW_BASE_URL}")
    st.write(f"**MySQL DB:** {MYSQL_DATABASE}")
    st.write(f"**MySQL Host:** {MYSQL_HOST}:{MYSQL_PORT}")

    try:
        conn = get_db_connection()
        conn.close()
        st.success("MySQL connection successful")
    except Error as e:
        st.error(f"MySQL connection failed: {e}")

    st.divider()
    st.subheader("Orders in DB")
    try:
        rows = get_all_orders(limit=50)
        if rows:
            st.dataframe(rows, use_container_width=True)
        else:
            st.info("No orders found in the table.")
    except Error as e:
        st.error(f"Could not load orders: {e}")



# ──────────────────────────────────────────────────────────────
# 9. MAIN UI
# ──────────────────────────────────────────────────────────────
st.title(" Order Support Chatbot")
st.caption("LangGraph + MemorySaver + MySQL + Streamlit")

for msg in st.session_state.chat_messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])


# ──────────────────────────────────────────────────────────────
# 10. CHAT HANDLER
# ──────────────────────────────────────────────────────────────
if user_prompt := st.chat_input("Ask about an order or refund..."):
    st.session_state.chat_messages.append({"role": "user", "content": user_prompt})
    with st.chat_message("user"):
        st.markdown(user_prompt)

    thread_cfg = {"configurable": {"thread_id": st.session_state.thread_id}}

    try:
        with st.chat_message("assistant"):
            with st.spinner("Thinking..."):
                result = app.invoke(
                    {"messages": [HumanMessage(content=user_prompt)]},
                    config=thread_cfg,
                )
                assistant_reply = result["messages"][-1].content
                st.markdown(assistant_reply)

        st.session_state.chat_messages.append({"role": "assistant", "content": assistant_reply})

    except Exception as e:
        error_msg = f"Sorry, something went wrong: {e}"
        with st.chat_message("assistant"):
            st.error(error_msg)
        st.session_state.chat_messages.append({"role": "assistant", "content": error_msg})


# ──────────────────────────────────────────────────────────────
# 11. OPTIONAL NOTES FOR RUNNING
# ──────────────────────────────────────────────────────────────
# Run with:
#   streamlit run app.py
#
# Example .env:
#   LLMGW_API_KEY=your_key_here
#   LLMGW_BASE_URL=https://llmgw-wp.tekstac.com
#   MYSQL_HOST=localhost
#   MYSQL_PORT=3306
#   MYSQL_USER=root
#   MYSQL_PASSWORD=your_password
#   MYSQL_DATABASE=support_bot
#
# Example SQL (if database not already created):
#   CREATE DATABASE support_bot;
#   USE support_bot;


In [ ]:
# ── Human-in-the-Loop (HITL) with interrupt_before ───────────────────────
print("\U0001f64b Human-in-the-Loop (HITL)")
print("=" * 55)
print()

# Compile agent_builder with interrupt_before + a fresh checkpointer
hitl_ckpt  = MemorySaver()
hitl_graph = agent_builder.compile(
    checkpointer=hitl_ckpt,
    interrupt_before=["tools"],      # \u2190 pause BEFORE the tools node
)

# ── Demo 1: Human APPROVES the tool call ─────────────────────────────────
cfg1   = {"configurable": {"thread_id": "hitl-approve"}}
query1 = "What is the square root of 625?"

print("\u2500" * 52)
print(f"\U0001f464 User   : {query1}")
print("\u2500" * 52)

# Run until the interrupt fires
paused = hitl_graph.invoke(
    {"messages": [HumanMessage(content=query1)]},
    config=cfg1,
)

last_msg = paused["messages"][-1]
print("\n\u23f8\ufe0f  PAUSED \u2014 waiting for human review of proposed tool call(s):\n")
if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
    for tc in last_msg.tool_calls:
        print(f"     Tool : {tc['name']}")
        print(f"     Args : {tc['args']}")

print("\n\u2705 Human decision: APPROVED \u2014 resuming execution...")
final = hitl_graph.invoke(None, config=cfg1)    # pass None to resume
print(f"\U0001f916 Final : {final['messages'][-1].content.strip()}")

# ── Demo 2: Human REJECTS the tool call ──────────────────────────────────
print()
print("\u2500" * 52)
cfg2   = {"configurable": {"thread_id": "hitl-reject"}}
query2 = "What is 100 plus 200?"

print(f"\U0001f464 User   : {query2}")
print("\u2500" * 52)

paused2 = hitl_graph.invoke(
    {"messages": [HumanMessage(content=query2)]},
    config=cfg2,
)

last2 = paused2["messages"][-1]
print("\n\u23f8\ufe0f  PAUSED \u2014 proposed tool call(s):")
if hasattr(last2, "tool_calls") and last2.tool_calls:
    for tc in last2.tool_calls:
        print(f"   Tool: {tc['name']}  Args: {tc['args']}")

print("\n\u274c Human decision: REJECTED \u2014 NOT resuming")
print("   Execution stops here. The paused state remains in the checkpointer.")
print("   (In production: log the rejection, notify the user, or update state before declining.)")


In [ ]:
#If you want real user approval there, use this pattern in that section:I(implemented in next cell)


decision = input("\nDo you approve this tool call? (approved/rejected): ").strip().lower()

if decision == "approved":
    print("\nApproved -> resuming...")
    final = hitl_graph.invoke(None, config=cfg1)
    print(f"Final: {final['messages'][-1].content.strip()}")
elif decision == "rejected":
    print("\nRejected -> not resuming.")
else:
    print("\nInvalid input. Enter approved or rejected.")

### Human-in-the-Loop (HITL) with `interrupt_before`

**HITL** pauses graph execution *before* a critical node so a human can review, approve, or reject the proposed action.

How it works:
1. Compile with `interrupt_before=["node_name"]` — execution pauses **before** that node runs
2. A **checkpointer is required** — the paused state must be stored until the human responds
3. Call `.invoke(None, config=config)` to **resume** from the saved checkpoint
4. To **reject** — simply do not call `.invoke(None, ...)` (or handle the state before resuming)

```
User input  →  [agent]  →  proposes tool call
                            ↓  ← PAUSE  (interrupt_before="tools")
            Human inspects state["messages"][-1].tool_calls
                            ↓ APPROVED
            [tools]  →  execute  →  [agent]  →  final answer
```

In [ ]:
# ── Human-in-the-Loop (HITL) with USER INPUT for approval/rejection ──────
print("🙋 Human-in-the-Loop (HITL) - Interactive User Decision")
print("=" * 55)
print()

# Reset with fresh checkpointer
hitl_ckpt_interactive = MemorySaver()
hitl_graph_interactive = agent_builder.compile(
    checkpointer=hitl_ckpt_interactive,
    interrupt_before=["tools"],
)

# Function to handle user approval/rejection
def hitl_interactive_demo(user_query: str, thread_id: str):
    """
    Run a single HITL demo with user input for approval/rejection.
    
    Args:
        user_query: The question to ask the agent
        thread_id:  Unique identifier for this session
    """
    cfg = {"configurable": {"thread_id": thread_id}}
    
    print("─" * 52)
    print(f"👤 User   : {user_query}")
    print("─" * 52)
    
    # Run until the interrupt fires
    paused = hitl_graph_interactive.invoke(
        {"messages": [HumanMessage(content=user_query)]},
        config=cfg,
    )
    
    last_msg = paused["messages"][-1]
    print("\n⏸️  PAUSED — proposed tool call(s):\n")
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        for tc in last_msg.tool_calls:
            print(f"     Tool : {tc['name']}")
            print(f"     Args : {tc['args']}")
    else:
        print("   (No tool calls)")
    
    # ASK THE USER FOR INPUT
    print()
    decision = input("❓ Do you approve this tool call? (approved/rejected): ").strip().lower()
    
    if decision == "approved":
        print("\n✅ Decision: APPROVED — resuming execution...")
        final = hitl_graph_interactive.invoke(None, config=cfg)
        print(f"🤖 Final : {final['messages'][-1].content.strip()}")
    elif decision == "rejected":
        print("\n❌ Decision: REJECTED — NOT resuming")
        print("   Execution stops here. The paused state remains in the checkpointer.")
    else:
        print(f"\n⚠️  Invalid input: '{decision}'. Please enter 'approved' or 'rejected'.")
    
    print()

# ── Run interactive demos ─────────────────────────────────────────────────
hitl_interactive_demo("What is the square root of 625?", "hitl-interactive-1")
hitl_interactive_demo("What is 100 plus 200?", "hitl-interactive-2")


In [ ]:
# ── Human-in-the-Loop Loan Approval System  ─────────────────────
print("🏦 Loan Approval System with Human Review")
print("=" * 60)
print()

@tool
def check_credit_score(applicant_id: str) -> str:
    """Check the applicant's credit score."""
    scores = {
        "APP-001": {"score": 750, "status": "Excellent"},
        "APP-002": {"score": 620, "status": "Fair"},
        "APP-003": {"score": 800, "status": "Excellent"},
    }
    data = scores.get(applicant_id, {"score": 700, "status": "Good"})
    return f"Credit Score: {data['score']} ({data['status']})"

@tool
def verify_income(applicant_id: str) -> str:
    """Verify annual income."""
    incomes = {
        "APP-001": 85000,
        "APP-002": 45000,
        "APP-003": 120000,
    }
    income = incomes.get(applicant_id, 70000)
    return f"Annual Income: ${income:,}"

@tool
def calculate_dti(applicant_id: str, loan_amount: float) -> str:
    """Calculate Debt-to-Income ratio using actual verified income."""
    incomes = {
        "APP-001": 85000,
        "APP-002": 45000,
        "APP-003": 120000,
    }
    annual_income = incomes.get(applicant_id, 70000)
    monthly_payment = (loan_amount / 60) + 500  # 5-year loan + existing debts
    monthly_income  = annual_income / 12
    dti_ratio = (monthly_payment / monthly_income) * 100
    status = "✅ PASS" if dti_ratio < 43 else "❌ FAIL"
    return f"Debt-to-Income Ratio: {dti_ratio:.1f}% {status} (max: 43%)"

@tool
def verify_employment(applicant_id: str) -> str:
    """Verify employment status."""
    employments = {
        "APP-001": "Employed - Tech Corp (5 years)",
        "APP-002": "Self-employed - Freelancer (2 years)",
        "APP-003": "Employed - Finance Co (8 years)",
    }
    return employments.get(applicant_id, "Employed (3 years)")

# ── Create Loan Approval Agent ───────────────────────────────────────────
loan_tools     = [check_credit_score, verify_income, calculate_dti, verify_employment]
tool_map_loans = {t.name: t for t in loan_tools}
llm_loans      = chat_model.bind_tools(loan_tools)

class LoanState(TypedDict):
    applicant_id: str
    loan_amount:  float
    messages:     Annotated[list[BaseMessage], operator.add]

def loan_agent(state: LoanState) -> dict:
    """Agent gathers loan info and makes a COMPREHENSIVE recommendation."""
    # ── FIX: Use SystemMessage so instructions are never duplicated in the
    #         message history and don't create consecutive HumanMessages. ──
    system_prompt = f"""You are a loan approval specialist. Analyze this loan application:
    - Applicant: {state['applicant_id']}
    - Loan Amount: ${state['loan_amount']:,.0f}

    Use ALL available tools to verify:
    1. Credit score (CIBIL/credit score)
    2. Annual income
    3. Debt-to-income ratio
    4. Employment status

    After gathering all information, provide a CLEAR RECOMMENDATION:

    Format your response like this:
    ════════════════════════════════════════════════════════
    RECOMMENDATION: [APPROVE / REJECT]

    Approval Factors:
    ✅ Credit Score: [score and status]
    ✅ Annual Income: [amount and assessment]
    ✅ Debt-to-Income Ratio: [percentage and status]
    ✅ Employment: [status and stability]

    Risk Assessment: [LOW / MEDIUM / HIGH]

    Reasoning:
    [Explain your decision clearly]
    ════════════════════════════════════════════════════════

    Be specific about WHY you approve or reject."""

    return {"messages": [llm_loans.invoke(
        [SystemMessage(content=system_prompt)] + state["messages"]
    )]}

def loan_tools_exec(state: LoanState) -> dict:
    """Execute verification tools."""
    last_msg = state["messages"][-1]
    results  = []
    for tc in last_msg.tool_calls:
        output = tool_map_loans[tc["name"]].invoke(tc["args"])
        results.append(ToolMessage(content=output, tool_call_id=tc["id"]))
    return {"messages": results}

def should_continue_loan(state: LoanState) -> Literal["loan_tools", "__end__"]:
    """Route: if tools needed, go to tools; else end."""
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "loan_tools"
    return END

# ── Build loan approval graph ─────────────────────────────────────────────
loan_builder = StateGraph(LoanState)
loan_builder.add_node("loan_agent", loan_agent)
loan_builder.add_node("loan_tools", loan_tools_exec)
loan_builder.add_edge(START, "loan_agent")
loan_builder.add_conditional_edges("loan_agent", should_continue_loan)
loan_builder.add_edge("loan_tools", "loan_agent")

# Pause BEFORE gathering sensitive financial info
loan_hitl = loan_builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["loan_tools"],
)

# ── Helper function to extract final answer ───────────────────────────────
def get_final_answer(result):
    """Extract the final answer from messages."""
    for msg in reversed(result["messages"]):
        if hasattr(msg, "content") and isinstance(msg.content, str):
            if msg.content.strip():
                return msg.content.strip()
    return "No recommendation found"

# ── Interactive Loan Approval Function ───────────────────────────────────
def process_loan_application(applicant_id: str, loan_amount: float):
    """Process loan with human approval at each step."""
    cfg = {"configurable": {"thread_id": f"loan-{applicant_id}"}}

    print("\n" + "─" * 60)
    print(f"📋 Application: {applicant_id}")
    print(f"💰 Loan Amount: ${loan_amount:,.0f}")
    print("─" * 60)
    print("\n🤖 Agent: Preparing to verify application...")

    paused = loan_hitl.invoke(
        {
            "applicant_id": applicant_id,
            "loan_amount":  loan_amount,
            # ── FIX: single HumanMessage — SystemMessage in loan_agent carries
            #         the instructions, so no consecutive HumanMessages occur. ──
            "messages": [HumanMessage(content="Please analyze this loan application and provide your recommendation.")],
        },
        config=cfg,
    )

    last_msg = paused["messages"][-1]
    print("\n⏸️  PAUSED - Will verify:")
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        for tc in last_msg.tool_calls:
            print(f"   • {tc['name']}")

    decision = input("\n👤 Loan Officer: Proceed with verification? (approved/rejected): ").strip().lower()

    if decision != "approved":
        print("\n❌ Application stopped by loan officer.")
        return

    print("\n✅ Verified! Gathering information...\n")
    final = loan_hitl.invoke(None, config=cfg)

    recommendation = get_final_answer(final)
    print("\n" + "─" * 60)
    print("🤖 AGENT RECOMMENDATION:")
    print("─" * 60)
    print(recommendation)

    print("\n" + "─" * 60)
    final_decision = input("👤 Loan Officer: Final decision? (approved/rejected): ").strip().lower()

    if final_decision == "approved":
        print(f"\n✅ LOAN APPROVED - {applicant_id}")
        print(f"   Amount: ${loan_amount:,.0f}")
    else:
        print(f"\n❌ LOAN REJECTED - {applicant_id}")
        print(f"   Amount: ${loan_amount:,.0f}")

    print("─" * 60 + "\n")

# ── RUN EXAMPLES ──────────────────────────────────────────────────────────
process_loan_application("APP-001", 250000)
process_loan_application("APP-002", 150000)

In [ ]:
from IPython.display import Image, display

# ── Graph 1: Cyclic Agent Graph ───────────────────────────────
print("=== Cyclic Agent Graph ===")
display(Image(agent_graph.get_graph().draw_mermaid_png()))

# ── Graph 2: Memory / Persistent Chat Graph ───────────────────
print("=== Memory (MemorySaver) Graph ===")
display(Image(persistent_chat.get_graph().draw_mermaid_png()))

# ── Graph 3: HITL Graph ───────────────────────────────────────
print("=== HITL Graph ===")
display(Image(hitl_graph.get_graph().draw_mermaid_png()))

# ── Graph 4: Loan Approval HITL Graph ────────────────────────
print("=== Loan Approval Graph ===")
display(Image(loan_hitl.get_graph().draw_mermaid_png()))